# Запуск экспериментов на суперкомпьютере

Ячейки с кодом — это **shell-команды**: копируйте их целиком в терминал кластера.
Все команды выполняются **из корня репозитория**.

Как устроено:
- каждый эксперимент — отдельный скрипт `exps/<name>.py` (параметры: `python exps/<name>.py --help`);
- длинные эксперименты разбиты на независимые задачи (по датасету и случаю unlabeled/labeled), их можно запускать параллельно;
- каждая задача пишет свой JSON в `results/exps/<эксперимент>/` и лог в `logs/<задача>.log`; JSON перезаписывается после каждого готового датасета или размера, поэтому прерванная задача не теряет готовые строки;
- в конце `exps/plot_results.py` склеивает все JSON одного эксперимента и строит графики с таблицами.

Оценки времени ниже ориентировочные: они экстраполированы с MUTAG (188 графов) и маленьких прогонов на ноутбуке и могут отличаться в 2–3 раза.

## 0. Подготовка (один раз)

### 0.1. Код и окружение
Скопируйте репозиторий на кластер (например, `rsync -av --exclude results --exclude tu_datasets ./ user@cluster:~/random_walk_kernels_computations/` с ноутбука) и создайте окружение. Нужен Python ≥ 3.11.

In [1]:
cd ~/random_walk_kernels_computations
python3 -m venv .venv
source .venv/bin/activate
pip install --upgrade pip
pip install numpy scipy networkx scikit-learn matplotlib

SyntaxError: invalid syntax (3924566678.py, line 1)

### 0.2. Проверка
Юнит-тесты и smoke-тесты всех экспериментов (около минуты).

In [ ]:
python -m unittest discover -s tests

### 0.3. Датасеты
Вычислительные узлы часто без интернета, поэтому скачайте TU-датасеты заранее на login-узле (в `tu_datasets/`).

In [ ]:
python -c "from exps.common import download_tu; [download_tu(n, 'tu_datasets') for n in ['MUTAG', 'ENZYMES', 'PTC_MR', 'AIDS', 'NCI1', 'ZINC_test']]"

### 0.4. Функция запуска `run`
`run <имя задачи> <скрипт> <аргументы>` запускает одну задачу, лог пишется в `logs/<имя>.log`.
BLAS ограничен одним потоком, чтобы время методов было сопоставимо.
Выполните **один** из двух вариантов в каждой новой сессии терминала (после `source .venv/bin/activate`).

**Вариант A — в фоне на текущем узле** (`nohup`; подходит для выделенного узла или интерактивной сессии):

In [ ]:
source .venv/bin/activate
export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1
mkdir -p logs
run() { name=$1; shift; nohup python "$@" > "logs/$name.log" 2>&1 & echo "$name: pid $!"; }

**Вариант B — SLURM** (каждая команда уходит отдельной задачей через `sbatch`). Поправьте под ваш кластер `--time`, `--mem` и, если нужно, добавьте `--partition=...`. Окружение (активированный `.venv`) передаётся в задачу автоматически.

In [ ]:
source .venv/bin/activate
mkdir -p logs
run() { name=$1; shift; sbatch --job-name="$name" --output="logs/$name.log" --cpus-per-task=1 --mem=32G --time=3-00:00:00 --wrap="export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1; python $*"; }

### 0.5. Мониторинг

In [ ]:
tail -n 2 logs/*.log          # последние строки всех задач
grep -l Traceback logs/*.log  # задачи, упавшие с ошибкой
squeue -u $USER               # (SLURM) что ещё в очереди или идёт

## 1. Scaling: время и ошибка от размера графа

Пары графов BA (m = 2), N = 8 … 8192 по степеням двойки, 3 повтора. Labeled: 3 метки рёбер, каждая с вероятностью 1/3. v и w — normal, λ = 0.7.

- Методы: Direct (N < 128), CG, FP, Sylvester (только unlabeled, N ≤ 512), GVoys (100 сэмплов), MCRWK с m = cN (c калибруется на N = 1024, чтобы время совпадало с GVoys) и с фиксированными m = 100, 1000, 10000.
- Ошибка считается относительно Direct при N < 128 и CG при N ≥ 128.
- Время: около 1–2 ч на случай; при N = 8192 CG и FP держат в памяти матрицы 8192 × 8192, это несколько ГБ.

In [ ]:
run scaling_unlabeled exps/scaling.py --cases unlabeled --experiment-name unlabeled
run scaling_labeled   exps/scaling.py --cases labeled   --experiment-name labeled

## 2. TU: SVM-классификация и SVR-регрессия

Классификация: MUTAG, ENZYMES, PTC_MR, AIDS, NCI1. Регрессия: ZINC_test. Датасеты полные. Labeled только там, где есть метки рёбер (MUTAG, PTC_MR, AIDS, ZINC_test).

- `evaluation` — SVC/SVR на Gram для всех методов, по этой колонке сравниваются методы;
- `evaluation_linear` — LinearSVC/LinearSVR на признаках GVoys и смещённого MCRWK (линейно по числу графов);
- MCRWK в двух версиях: несмещённая и смещённая (`_biased`, PSD, работает как ridge);
- ошибка Gram относительно Direct, время признаков и Gram отдельно.

Время: MUTAG и PTC_MR — минуты; ENZYMES — около часа; AIDS — часы; NCI1 и ZINC_test — до суток и больше (основное время уходит на точные методы: Direct, CG, FP, Sylvester).

In [ ]:
for d in MUTAG PTC_MR ENZYMES AIDS NCI1 ZINC_test; do
  run tu_svm_${d}_unlabeled exps/tu_svm.py --datasets $d --cases unlabeled --experiment-name ${d}_unlabeled
done
for d in MUTAG PTC_MR AIDS ZINC_test; do
  run tu_svm_${d}_labeled exps/tu_svm.py --datasets $d --cases labeled --experiment-name ${d}_labeled
done

## 3. Время построения Gram

Время построения Gram (без SVM) на полных датасетах классификации: MUTAG, PTC_MR, ENZYMES, AIDS, NCI1.

У GVoys и MCRWK время записывается по частям:
- `feature_time_sec` — построение признаков, до вычисления Gram;
- `gram_build_sec` — само произведение для Gram;
- `time_sec` — их сумма, она сравнима со временем точных методов.

Графики: полное время Gram всех методов (`gram_time_<case>.png`) и отдельно время построения признаков GVoys и MCRWK (`gram_time_features_<case>.png`).

Время: MUTAG и PTC_MR — минуты, точные методы на AIDS и NCI1 — часы.

In [ ]:
for d in MUTAG PTC_MR ENZYMES AIDS NCI1; do
  run gram_time_${d}_unlabeled exps/gram_time.py --datasets $d --cases unlabeled --experiment-name ${d}_unlabeled
done
for d in MUTAG PTC_MR AIDS; do
  run gram_time_${d}_labeled exps/gram_time.py --datasets $d --cases labeled --experiment-name ${d}_labeled
done

## 4. Влияние λ (0.1 … 0.9)

- **Synthetic:** пары графов N = 256; время и ошибка от λ.
- **TU:** время Gram, ошибка и SVM accuracy от λ на пяти датасетах классификации.

c для m = cN калибруется заново для каждого λ.

TU-часть — это девять прогонов, по одному на каждое λ, каждый стоит как эксперимент 2, так что NCI1 и AIDS очень долгие. Если нужно быстрее, добавьте `--max-graphs 1000` (стратифицированная подвыборка).

In [ ]:
run lambda_synthetic_unlabeled exps/lambda_sweep.py --settings synthetic --cases unlabeled --experiment-name synthetic_unlabeled
run lambda_synthetic_labeled   exps/lambda_sweep.py --settings synthetic --cases labeled   --experiment-name synthetic_labeled
for d in MUTAG PTC_MR ENZYMES AIDS NCI1; do
  run lambda_tu_${d}_unlabeled exps/lambda_sweep.py --settings tu --datasets $d --cases unlabeled --experiment-name tu_${d}_unlabeled
done
for d in MUTAG PTC_MR AIDS; do
  run lambda_tu_${d}_labeled exps/lambda_sweep.py --settings tu --datasets $d --cases labeled --experiment-name tu_${d}_labeled
done

## 5. Распределение меток q (importance sampling)

Только labeled: MUTAG, PTC_MR, AIDS, ZINC_test. Сравниваются предложения uniform, q ∝ p^α (`freq:0.5`, `freq:1`, `freq:2`), `sq_mean` (mean-field оптимум), `norm_l1`, `norm_fro`, `inverse` (плохой контроль) и `random`. Сетка: m = 100, 1000, 10000, λ = 0.3 и 0.7, 5 сидов на точку.

Что записывается:
- по каждому сиду — время, ошибка, диагностика блужданий и SVM;
- по конфигурации — bias, дисперсия и MSE по элементам Gram, MSE × время, ridge τ/K, R(q).

Время: MUTAG и PTC_MR — около часа; AIDS и ZINC_test — много часов (SVM на каждый сид; `--skip-evaluation` сильно ускоряет).

In [ ]:
for d in MUTAG PTC_MR AIDS ZINC_test; do
  run q_sampling_${d} exps/q_sampling.py --datasets $d --experiment-name $d
done

## 6. Число последовательностей меток на длину n

n = 1, 2, 4, 8, 16 в двух дизайнах:
- фиксированный бюджет B = m·n (B = 1000 и 10000); по статье здесь оптимально n = 1;
- фиксированное число длин m = 1000; выигрыш от n должен насыщаться.

Рядом записывается граница (10) и оба её слагаемых. Датасеты те же, что в эксперименте 5, λ = 0.7, 5 сидов.

In [ ]:
for d in MUTAG PTC_MR AIDS ZINC_test; do
  run n_sampling_${d} exps/n_sampling.py --datasets $d --experiment-name $d
done

## 7. Проверка теории: сходимость и границы

По 8 графов BA каждого размера N = 16, 64, 256, 1024 (28 пар), 50 сидов, m = 10 … 10000, λ = 0.3 и 0.7. Эталон: Direct или CG.

- относительный RMSE от m: наклон −½, кривые разных N должны совпадать;
- измеренная дисперсия пары против границ (6) и (10);
- доля пар с |k̂ − k| > εk против Хёфдинга (7) и Чебышёва (11);
- нужное m против (8) и (12).

При λ = 0.7 граница для labeled бесконечна (λ ≥ 1/3), в JSON она записана как null. Время — около часа на задачу.

In [ ]:
for c in unlabeled labeled; do
  for l in 0.3 0.7; do
    run convergence_${c}_${l} exps/convergence.py --cases $c --lambdas $l --experiment-name ${c}_${l}
  done
done

## 8. Ridge: одна модель на Gram и на признаках

Датасеты и методы как в эксперименте 2, но вместо SVM используется ridge:
- `evaluation` — kernel ridge на Gram, для всех методов;
- `evaluation_linear` — ridge на признаках GVoys и смещённого MCRWK. Это та же модель, что и kernel ridge на XXᵀ; `primal_dual_gap` проверяет совпадение численно.

Классификация — least squares на кодах ±1, регрессия — на стандартизированных таргетах. α подбирается во вложенной кросс-валидации.

Время: как в эксперименте 2, но само обучение быстрее SVM.

In [ ]:
for d in MUTAG PTC_MR ENZYMES AIDS NCI1 ZINC_test; do
  run ridge_${d}_unlabeled exps/ridge.py --datasets $d --cases unlabeled --experiment-name ${d}_unlabeled
done
for d in MUTAG PTC_MR AIDS ZINC_test; do
  run ridge_${d}_labeled exps/ridge.py --datasets $d --cases labeled --experiment-name ${d}_labeled
done

## 9. Графики и таблицы

Когда задачи закончились (можно и по частям — графики строятся по тем JSON, что уже есть):

In [ ]:
python exps/plot_results.py results/exps/*/*.json --out-dir fig/exps

Скопировать результаты с кластера на ноутбук (выполняется **на ноутбуке**, `user@cluster` замените на свой адрес):

In [ ]:
rsync -av user@cluster:~/random_walk_kernels_computations/results/exps/ ./results/exps/
rsync -av user@cluster:~/random_walk_kernels_computations/fig/exps/ ./fig/exps/

## Быстрая проверка (секунды)

Если хочется убедиться, что всё запускается, прежде чем ставить длинные задачи:

In [ ]:
python exps/scaling.py --sizes 8 16 --n-repeats 1 --n-samples-gvoys 5 --mc-fixed-m 100 --calibration-n 8 --calibration-repeats 1 --output-dir results/quick
python exps/tu_svm.py --datasets MUTAG ZINC_test --max-graphs 12 --n-splits 2 --inner-splits 2 --n-samples-gvoys 5 --mc-fixed-m 100 --calibration-graphs 3 --calibration-repeats 1 --output-dir results/quick
python exps/convergence.py --sizes 8 16 --n-graphs 3 --n-repeats 5 --m-values 10 100 --lambdas 0.3 --output-dir results/quick
python exps/ridge.py --datasets MUTAG ZINC_test --max-graphs 12 --n-splits 2 --inner-splits 2 --n-samples-gvoys 5 --mc-fixed-m 100 --calibration-graphs 3 --calibration-repeats 1 --output-dir results/quick